# Task 2.3 — Hyperparameter Experimentation

**Goal:** Compare learning rates, batch sizes, optimizers, and epoch counts and record RMSE/R².

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from itertools import product

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("../student_performance.csv")
X = df.drop(columns=["final_score"]).values.astype(np.float32)
y = df["final_score"].values.astype(np.float32).reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self,x):
        return self.net(x)

def run_experiment(lr, batch_size, optimizer_name, epochs):
    torch.manual_seed(42)
    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    model = Net()
    if optimizer_name == "Adam":
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_test)).numpy()
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    return rmse, r2


In [ ]:
learning_rates = [0.0001, 0.001, 0.01]
batch_sizes = [8, 16, 32]
optimizers = ["Adam", "SGD"]
epoch_options = [20, 50, 100]

results = []
# A compact experiment grid for practical Colab runtime.
settings = list(product(learning_rates, batch_sizes, optimizers, epoch_options))

for lr, bs, opt_name, ep in settings:
    rmse, r2 = run_experiment(lr, bs, opt_name, ep)
    results.append({
        "learning_rate": lr,
        "batch_size": bs,
        "optimizer": opt_name,
        "epochs": ep,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df.head(10)


In [ ]:
results_df.to_csv("hyperparameter_results.csv", index=False)
print("Saved hyperparameter_results.csv")

print("\nBest 10 configurations by RMSE:")
display(results_df.head(10))
